# Pipeline Gardenia | Etapa 7.5: Consultas em Spark SQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anlbp/Projeto-Gardenia-unifeob/blob/main/recursos/dados/notebook_dados_nao_filtrados/07_5_spark_sql.ipynb)

Complemento do notebook 07. Ele usa a mesma carga e a mesma tabela de fatos, mas faz as análises com **SQL** em vez de DataFrames. No Spark, as duas formas usam o mesmo motor (o otimizador Catalyst), então o resultado é o mesmo; o SQL é só outra forma de escrever.

**O que o notebook faz**

1. Lê os `trusted_*_N.csv` de todos os lotes (como no notebook 07).
2. Registra a tabela de fatos como a **view temporária** `fato_vendas`.
3. Roda 4 consultas que usam recursos de SQL analítico:

| Consulta | Recurso de SQL |
|---|---|
| Receita por trimestre e categoria, com a participação de cada categoria | função de janela com `SUM(...) OVER (PARTITION BY ...)` |
| Receita mensal e variação em relação ao mês anterior | CTE (`WITH`) e `LAG` |
| Os 3 produtos de maior receita em cada categoria | `ROW_NUMBER() OVER (PARTITION BY ...)` e subconsulta |
| Cancelamento por faixa etária | `CASE WHEN` e `HAVING` |

4. Mostra o plano de execução de uma consulta (`EXPLAIN`) e confere duas consultas contra a versão em DataFrame.
5. Grava os resultados em CSV (`sql_*.csv`).

**Lê (por upload):** o(s) `saida_etapa3_loteN.zip` do notebook 03, ou os `trusted_*_N.csv`.
**Gera:** `sql_*.csv`; ao final o navegador baixa sozinho o `spark_sql_etapa7_5.zip`.

## 1. Configuração

In [ ]:
import os
# ============================ CONFIGURAÇÃO ============================
# Fonte de dados: SOMENTE upload (drive, github e local estão desativados).
LOTES = None                # None = todos os lotes enviados. Ou uma lista, ex.: [1, 2]
PASTA_DADOS = "dados_trusted"   # pasta local do Colab onde os trusted enviados são gravados

# --- saída ---
SAIDA = "spark_saida"        # pasta local do Colab onde os resultados são gravados
BAIXAR_ZIP = True           # baixa automaticamente o spark_sql_etapa7_5.zip ao final
# =======================================================================

# --- Destino dos Gráficos (VM 3) ---
PASTA_GRAFICOS = os.environ.get("GRAFICOS_VM3_DIR", os.environ.get("GRAFICOS_DIR", "recursos/dados/graficos/vm3"))
os.makedirs(PASTA_GRAFICOS, exist_ok=True)

def salvar_grafico(fig, nome_arquivo):
    caminho = os.path.join(PASTA_GRAFICOS, nome_arquivo)
    fig.savefig(caminho, bbox_inches="tight", dpi=150)
    print(f"[Grafico salvo na VM3] {caminho}")

## 2. Instalação do Spark e leitura dos dados

In [ ]:
# O Spark precisa de Java 17 ou mais novo. No Colab, instala (leva cerca de 1 minuto na primeira vez).
import os, shutil, subprocess, sys

if shutil.which("java") is None or "17" not in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark
print("pyspark", pyspark.__version__)

pyspark 4.0.5


## 3. Código de apoio

As funções de carga e a tabela de fatos são as mesmas do notebook 07. As consultas SQL ficam no dicionário `CONSULTAS_SQL`, com uma descrição de cada uma.

In [ ]:
import io
import os
import re
import zipfile


def receber_upload(mensagem, pasta=None):
    """Abre o seletor de arquivos do Colab. Aceita arquivos soltos ou .zip (o .zip baixado pela etapa anterior).
    Devolve {nome_do_arquivo: bytes}. Se `pasta` for dada, também grava os arquivos nela.
    Nomes repetidos que o Colab renomeia (ex.: 'arquivo (1).csv') voltam ao nome original, valendo o último enviado."""
    from google.colab import files
    print(mensagem)
    enviados = files.upload()
    arquivos = {}
    for nome, dados in enviados.items():
        if nome.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(dados)) as z:
                for info in z.infolist():
                    if not info.is_dir():
                        arquivos[os.path.basename(info.filename)] = z.read(info)
        else:
            arquivos[nome] = dados
    arquivos = {re.sub(r" \(\d+\)(?=\.[^.]+$)", "", os.path.basename(n)): d for n, d in arquivos.items()}
    if pasta:
        os.makedirs(pasta, exist_ok=True)
        for n, d in arquivos.items():
            with open(os.path.join(pasta, n), "wb") as f:
                f.write(d)
    print("Recebidos:", ", ".join(sorted(arquivos)) or "(nenhum)")
    return arquivos


def baixar_zip(pasta, nome_zip):
    """Compacta a pasta e inicia o download automático no navegador (Colab)."""
    import shutil
    arquivo = shutil.make_archive(nome_zip, "zip", pasta)
    try:
        from google.colab import files
        files.download(arquivo)
        print("Download iniciado:", arquivo)
    except ImportError:
        print("Fora do Colab: arquivo gerado em", arquivo)
    return arquivo

import glob
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [ ]:
TABELAS = ["clientes", "produtos", "compras"]
COR_SEGMENTO = {"VIP": "#C44E52", "Regular": "#DD8452", "Baixo engajamento": "#4C72B0"}


def criar_spark(master="local[*]", memoria=None, particoes=8):
    """Sessão Spark. master="local[*]" usa todos os núcleos da máquina; num cluster seria "spark://host:7077"."""
    b = (SparkSession.builder.master(master).appName("gardenia-etapa7")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", str(particoes))
         .config("spark.sql.execution.arrow.pyspark.enabled", "false"))
    if memoria:
        b = b.config("spark.driver.memory", memoria)
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def lotes_trusted(pasta):
    """Lotes que podem ser lidos: os 3 trusted_*_N.csv existem e, se a pasta usa marcadores
    (trusted_N.pronto, gravados pela VM 2), o marcador também existe."""
    arquivos = set(os.listdir(pasta)) if os.path.isdir(pasta) else set()
    usa_marcador = any(re.fullmatch(r"trusted_\d+\.pronto", n) for n in arquivos)
    lotes = []
    for n in arquivos:
        m = re.fullmatch(r"trusted_clientes_(\d+)\.csv", n)
        if not m:
            continue
        N = int(m.group(1))
        if not all(f"trusted_{t}_{N}.csv" in arquivos for t in TABELAS):
            continue
        if usa_marcador and f"trusted_{N}.pronto" not in arquivos:
            continue
        lotes.append(N)
    return sorted(lotes)


def ler_tabela(spark, pasta, tabela, lotes):
    """Lê trusted_<tabela>_N.csv de cada lote e empilha, com a coluna 'lote'."""
    partes = []
    for n in lotes:
        caminho = os.path.abspath(os.path.join(pasta, f"trusted_{tabela}_{n}.csv"))
        partes.append(spark.read.csv(caminho, header=True, inferSchema=False).withColumn("lote", F.lit(n)))
    df = partes[0]
    for p in partes[1:]:
        df = df.unionByName(p)
    return df


def carregar(spark, pasta, lotes):
    """Devolve clientes, produtos e compras já tipados. A chave de cada registro é (lote, id),
    porque o simulador reinicia os ids a cada ciclo."""
    cli = (ler_tabela(spark, pasta, "clientes", lotes)
           .select("lote", F.col("id").cast("int").alias("cliente_id"), "nome",
                   F.col("idade").cast("int").alias("idade"),
                   F.to_timestamp("data_cadastro").alias("data_cadastro"),
                   (F.col("cadastro_incompleto") == "True").alias("cadastro_incompleto")))
    cli = cli.withColumn("faixa_etaria", F.when(F.col("idade").isNull(), "sem idade")
                         .when(F.col("idade") < 25, "18-24").when(F.col("idade") < 35, "25-34")
                         .when(F.col("idade") < 45, "35-44").when(F.col("idade") < 55, "45-54")
                         .when(F.col("idade") < 65, "55-64").otherwise("65+"))
    prod = (ler_tabela(spark, pasta, "produtos", lotes)
            .select("lote", F.col("id").cast("int").alias("produto_id"), F.col("nome").alias("produto"),
                    F.coalesce(F.col("categoria"), F.lit("Não informada")).alias("categoria"),   # categoria inválida foi anulada na limpeza
                    F.col("preco").cast("double").alias("preco"),
                    F.col("satisfacao_media").cast("double").alias("satisfacao_media")))
    comp = (ler_tabela(spark, pasta, "compras", lotes)
            .select("lote", F.col("id").cast("int").alias("compra_id"),
                    F.col("cliente_id").cast("int").alias("cliente_id"),
                    F.col("produto_id").cast("int").alias("produto_id"),
                    F.col("quantidade").cast("int").alias("quantidade"),
                    F.col("valor_total").cast("double").alias("valor_total"),
                    F.col("desconto").cast("double").alias("desconto"),
                    F.to_timestamp("data_compra").alias("data_compra"),
                    "status", F.col("satisfacao").cast("int").alias("satisfacao")))
    return cli, prod, comp


def montar_fato(cli, prod, comp):
    """Tabela de fatos: cada compra com os dados do produto e do cliente (join por lote + id).
    No simulador, `desconto` é um PERCENTUAL (0, 5, 10, 15) e `valor_total` já é o valor LÍQUIDO:
    valor_liquido = valor_total e valor_bruto = valor_liquido / (1 - desconto/100).
    Pedidos cancelados têm quantidade 0 e valor 0: não somam receita, mas contam na taxa de cancelamento."""
    fato = (comp.join(prod.select("lote", "produto_id", "produto", "categoria", "preco"), ["lote", "produto_id"], "inner")
                .join(cli.select("lote", "cliente_id", "idade", "faixa_etaria"), ["lote", "cliente_id"], "inner")
                .withColumn("valor_liquido", F.col("valor_total"))
                .withColumn("valor_bruto", F.round(F.col("valor_total") / (1 - F.col("desconto") / 100), 2))
                .withColumn("mes", F.date_format("data_compra", "yyyy-MM"))
                .withColumn("valida", F.col("status") != "cancelado"))
    return fato

In [ ]:
def pdf(df):
    return df.toPandas()

In [ ]:
CONSULTAS_SQL = {
    "sql_receita_trimestre_categoria": (
        "Receita líquida por trimestre e categoria, com a participação de cada categoria no trimestre (função de janela).",
        """
        SELECT trimestre, categoria,
               ROUND(SUM(valor_liquido), 2) AS receita_liquida,
               ROUND(100 * SUM(valor_liquido) / SUM(SUM(valor_liquido)) OVER (PARTITION BY trimestre), 1) AS pct_do_trimestre
        FROM fato_vendas
        WHERE valida
        GROUP BY trimestre, categoria
        ORDER BY trimestre, receita_liquida DESC
        """),
    "sql_variacao_mensal": (
        "Receita mensal e variação percentual em relação ao mês anterior (LAG).",
        """
        WITH mensal AS (
            SELECT mes, SUM(valor_liquido) AS receita
            FROM fato_vendas WHERE valida GROUP BY mes
        )
        SELECT mes, ROUND(receita, 2) AS receita_liquida,
               ROUND(100 * (receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes), 1) AS variacao_pct
        FROM mensal ORDER BY mes
        """),
    "sql_top3_produtos_por_categoria": (
        "Os 3 produtos de maior receita em cada categoria (ROW_NUMBER por categoria).",
        """
        SELECT * FROM (
            SELECT categoria, lote, produto_id, produto,
                   ROUND(SUM(valor_liquido), 2) AS receita_liquida,
                   ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(valor_liquido) DESC) AS posicao
            FROM fato_vendas WHERE valida
            GROUP BY categoria, lote, produto_id, produto
        ) WHERE posicao <= 3
        ORDER BY categoria, posicao
        """),
    "sql_cancelamento_faixa_etaria": (
        "Taxa de cancelamento por faixa etária (pedidos com status cancelado, quantidade 0 e valor 0).",
        """
        SELECT faixa_etaria,
               COUNT(*) AS compras,
               SUM(CASE WHEN valida THEN 0 ELSE 1 END) AS canceladas,
               ROUND(100 * AVG(CASE WHEN valida THEN 0.0 ELSE 1.0 END), 1) AS pct_cancelamento
        FROM fato_vendas
        GROUP BY faixa_etaria
        HAVING COUNT(*) >= 1
        ORDER BY pct_cancelamento DESC
        """),
}


def consultas_sql(fato):
    """Registra a tabela de fatos como view temporária e roda as consultas em Spark SQL."""
    spark = fato.sparkSession
    fato.withColumn("trimestre", F.concat(F.year("data_compra").cast("string"), F.lit("-T"),
                                          F.quarter("data_compra").cast("string"))) \
        .createOrReplaceTempView("fato_vendas")
    return {nome: pdf(spark.sql(sql)) for nome, (_, sql) in CONSULTAS_SQL.items()}

## 4. Carga e tabela de fatos

In [ ]:
import shutil

# upload dos trusted (um ou vários lotes; aceita o .zip baixado pelo notebook 03)
shutil.rmtree(PASTA_DADOS, ignore_errors=True)
receber_upload("Envie o saida_etapa3_lote*.zip (ou os arquivos trusted_clientes/produtos/compras_N.csv) do notebook 03", pasta=PASTA_DADOS)

spark = criar_spark("local[*]")
lotes = lotes_trusted(PASTA_DADOS)
if LOTES is not None:
    lotes = [n for n in lotes if n in LOTES]
if not lotes:
    raise FileNotFoundError(
        f"Não encontrei lotes completos em {PASTA_DADOS}.\n"
        "Rode antes o notebook 03 (ele grava trusted_clientes/produtos/compras_N.csv) e envie o .zip dele.")
print("Lotes lidos:", lotes)

cli, prod, comp = carregar(spark, PASTA_DADOS, lotes)
fato = montar_fato(cli, prod, comp).cache()
print(f"fato_vendas: {fato.count()} compras")
fato.printSchema()

Envie o saida_etapa3_lote*.zip (ou os arquivos trusted_clientes/produtos/compras_N.csv) do notebook 03


Saving saida_etapa3_lote1.zip to saida_etapa3_lote1.zip
Recebidos: quarentena_compras_1.csv, quarentena_produtos_1.csv, relatorio_qualidade_1.csv, trusted_clientes_1.csv, trusted_compras_1.csv, trusted_produtos_1.csv
Lotes lidos: [1]
fato_vendas: 330 compras
root
 |-- lote: integer (nullable = false)
 |-- cliente_id: integer (nullable = true)
 |-- produto_id: integer (nullable = true)
 |-- compra_id: integer (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- desconto: double (nullable = true)
 |-- data_compra: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- satisfacao: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = false)
 |-- preco: double (nullable = true)
 |-- idade: integer (nullable = true)
 |-- faixa_etaria: string (nullable = false)
 |-- valor_liquido: double (nullable = true)
 |-- valor_bruto: double (nullable = true)
 |-- mes: string (nullable =

## 5. As consultas

`consultas_sql` registra a view `fato_vendas` (acrescentando a coluna `trimestre`) e roda as 4 consultas. Para cada uma o notebook mostra a descrição, o SQL e o resultado.

In [ ]:
import inspect
sql_res = consultas_sql(fato)
for nome, d in sql_res.items():
    descricao, sql = CONSULTAS_SQL[nome]
    print(f"\n=== {nome}\n{descricao}\n{inspect.cleandoc(sql)}\n")
    display(d.head(15))


=== sql_receita_trimestre_categoria
Receita líquida por trimestre e categoria, com a participação de cada categoria no trimestre (função de janela).
SELECT trimestre, categoria,
       ROUND(SUM(valor_liquido), 2) AS receita_liquida,
       ROUND(100 * SUM(valor_liquido) / SUM(SUM(valor_liquido)) OVER (PARTITION BY trimestre), 1) AS pct_do_trimestre
FROM fato_vendas
WHERE valida
GROUP BY trimestre, categoria
ORDER BY trimestre, receita_liquida DESC



,trimestre,categoria,receita_liquida,pct_do_trimestre
0,2025-T4,Esportes,14033.79,45.6
1,2025-T4,Não informada,9038.79,29.3
2,2025-T4,Roupas,7733.45,25.1
3,2026-T1,Esportes,15793.06,45.0
4,2026-T1,Não informada,10409.52,29.7
5,2026-T1,Roupas,8855.50,25.3
6,2026-T2,Esportes,31492.80,55.7
7,2026-T2,Não informada,13688.94,24.2
8,2026-T2,Roupas,11370.85,20.1
9,2026-T3,Esportes,12723.43,36.6



=== sql_variacao_mensal
Receita mensal e variação percentual em relação ao mês anterior (LAG).
WITH mensal AS (
    SELECT mes, SUM(valor_liquido) AS receita
    FROM fato_vendas WHERE valida GROUP BY mes
)
SELECT mes, ROUND(receita, 2) AS receita_liquida,
       ROUND(100 * (receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes), 1) AS variacao_pct
FROM mensal ORDER BY mes



,mes,receita_liquida,variacao_pct
0,2025-11,9967.97,NaN
1,2025-12,20838.06,109.1
2,2026-01,10538.13,-49.4
3,2026-02,12278.19,16.5
4,2026-03,12241.76,-0.3
5,2026-04,14967.80,22.3
6,2026-05,23665.24,58.1
7,2026-06,17919.55,-24.3
8,2026-07,11740.76,-34.5
9,2026-08,15368.08,30.9



=== sql_top3_produtos_por_categoria
Os 3 produtos de maior receita em cada categoria (ROW_NUMBER por categoria).
SELECT * FROM (
    SELECT categoria, lote, produto_id, produto,
           ROUND(SUM(valor_liquido), 2) AS receita_liquida,
           ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(valor_liquido) DESC) AS posicao
    FROM fato_vendas WHERE valida
    GROUP BY categoria, lote, produto_id, produto
) WHERE posicao <= 3
ORDER BY categoria, posicao



,categoria,lote,produto_id,produto,receita_liquida,posicao
0,Esportes,1,116,Raquete Classic,3648.27,1
1,Esportes,1,319,Meia Preto,3381.12,2
2,Esportes,1,468,Raquete Slim,3228.16,3
3,Não informada,1,61,Moletom Preto,4301.82,1
4,Não informada,1,473,Bermuda Branco,2964.69,2
5,Não informada,1,471,Moletom Branco,2654.73,3
6,Roupas,1,447,Camisa Cinza,2280.61,1
7,Roupas,1,112,Moletom Pro,2207.19,2
8,Roupas,1,189,Jaqueta Azul,1850.78,3



=== sql_cancelamento_faixa_etaria
Taxa de cancelamento por faixa etária (pedidos com status cancelado, quantidade 0 e valor 0).
SELECT faixa_etaria,
       COUNT(*) AS compras,
       SUM(CASE WHEN valida THEN 0 ELSE 1 END) AS canceladas,
       ROUND(100 * AVG(CASE WHEN valida THEN 0.0 ELSE 1.0 END), 1) AS pct_cancelamento
FROM fato_vendas
GROUP BY faixa_etaria
HAVING COUNT(*) >= 1
ORDER BY pct_cancelamento DESC



,faixa_etaria,compras,canceladas,pct_cancelamento
0,45-54,40,17,42.5
1,35-44,51,18,35.3
2,18-24,35,11,31.4
3,65+,86,22,25.6
4,55-64,62,11,17.7
5,25-34,56,8,14.3


## 6. Interpretação das consultas

Preencha depois de rodar, com os números que apareceram:

- **Trimestre e categoria:** qual categoria lidera em cada trimestre e se a participação muda ou fica parecida (com 2 categorias, perto de 50% cada uma significa que nenhuma domina).
- **Variação mensal:** os meses com maior alta e maior queda. O primeiro mês não tem variação (não há mês anterior), e o primeiro e o último mês costumam estar incompletos.
- **Top 3 por categoria:** se os mesmos produtos aparecem com frequência ou se a receita é pulverizada.
- **Cancelamento por faixa etária:** se a diferença entre faixas é grande o bastante para agir (confira com o teste do notebook 08).

## 7. Plano de execução e conferência

In [ ]:
# O mesmo motor executa SQL e DataFrames. EXPLAIN mostra o plano que o Spark montou (leia de baixo para cima).
# Extrai o plano textual formatado sem encapsular na tabela ASCII que causa overflow visual
plano_res = spark.sql("EXPLAIN FORMATTED " + inspect.cleandoc(CONSULTAS_SQL["sql_cancelamento_faixa_etaria"][1])).collect()[0][0]
print(plano_res)


== Physical Plan ==
AdaptiveSparkPlan (42)
+- Sort (41)
   +- Exchange (40)
      +- Filter (39)
         +- HashAggregate (38)
            +- Exchange (37)
               +- HashAggregate (36)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                              +- AdaptiveSparkPlan (35)
                                 +- == Final Plan ==
                                    ResultQueryStage (21)
                                    +- * Project (20)
                                       +- * BroadcastHashJoin Inner BuildRight (19)
                                          :- * Project (12)
                                          :  +- * BroadcastHashJoin Inner BuildLeft (11)
                                          :     :- BroadcastQueryStage (7), Statistics(sizeInBytes=1027.9 KiB, rowCount=330)
                                          :     :  +- BroadcastExchange (6)
                                          :     :     +- * P

In [ ]:
# Conferência: a mesma conta feita com DataFrames precisa dar o mesmo resultado que o SQL.
from pyspark.sql import functions as F

df_canc = (fato.groupBy("faixa_etaria")
           .agg(F.count("*").alias("compras"),
                F.sum(F.when(F.col("valida"), 0).otherwise(1)).alias("canceladas"),
                F.round(100 * F.avg(F.when(F.col("valida"), 0.0).otherwise(1.0)), 1).alias("pct_cancelamento"))
           .filter("compras >= 1").toPandas())
sql_canc = sql_res["sql_cancelamento_faixa_etaria"]
a = df_canc.sort_values("faixa_etaria").reset_index(drop=True)
b = sql_canc.sort_values("faixa_etaria").reset_index(drop=True)[a.columns]
assert a.equals(b) or (a.astype(str).values == b.astype(str).values).all(), "SQL e DataFrame divergem"

mensal_df = (fato.filter("valida").groupBy("mes").agg(F.round(F.sum("valor_liquido"), 2).alias("receita_liquida"))
             .orderBy("mes").toPandas())
mensal_sql = sql_res["sql_variacao_mensal"][["mes", "receita_liquida"]]
assert np.allclose(mensal_df["receita_liquida"], mensal_sql["receita_liquida"]), "receita mensal diverge"
print("Conferência: SQL e DataFrame dão o mesmo resultado (cancelamento por faixa e receita mensal).")

Conferência: SQL e DataFrame dão o mesmo resultado (cancelamento por faixa e receita mensal).


## 8. Salvar resultados

In [ ]:
os.makedirs(SAIDA, exist_ok=True)
for nome, d in sql_res.items():
    destino = os.path.join(SAIDA, f"{nome}.csv")
    d.to_csv(destino, index=False)
    print(f"  {destino}  ({len(d)} linhas)")
spark.stop()

# download automático do .zip
if BAIXAR_ZIP:
    baixar_zip(SAIDA, "spark_sql_etapa7_5")

  spark_saida/sql_receita_trimestre_categoria.csv  (15 linhas)
  spark_saida/sql_variacao_mensal.csv  (13 linhas)
  spark_saida/sql_top3_produtos_por_categoria.csv  (9 linhas)
  spark_saida/sql_cancelamento_faixa_etaria.csv  (6 linhas)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download iniciado: /content/spark_sql_etapa7_5.zip
